In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def safe_alpha_name(alpha: float) -> str:
    return f"{float(alpha):.2f}".replace("-", "m").replace(".", "p")


def plot_cp_curve(df: pd.DataFrame, alpha_value: float, output_path: str | Path, title_prefix: str = "PC-SSOR") -> None:
    sub = df[df["Alpha"] == alpha_value].copy()
    if len(sub) == 0:
        return
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(7, 5))
    for side in ["upper", "lower"]:
        s = sub[sub["side"] == side].sort_values("x")
        if len(s) == 0:
            continue
        if "Cp_pred" in s.columns:
            plt.plot(s["x"].values, s["Cp_pred"].values, marker="o", linestyle="-", label=f"Pred {side}")
        if "Cp" in s.columns:
            plt.scatter(s["x"].values, s["Cp"].values, s=24, label=f"True {side}")
        if "Cp_lo_90" in s.columns and "Cp_hi_90" in s.columns:
            plt.fill_between(s["x"].values, s["Cp_lo_90"].values, s["Cp_hi_90"].values, alpha=0.15)
    plt.gca().invert_yaxis()
    plt.xlabel("x/c")
    plt.ylabel("Cp")
    plt.title(f"{title_prefix} | alpha={float(alpha_value):.2f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=250, bbox_inches="tight")
    plt.close()


def save_per_alpha_curves(df: pd.DataFrame, csv_dir: str | Path, plot_dir: str | Path, title_prefix: str = "PC-SSOR") -> None:
    csv_dir = Path(csv_dir)
    plot_dir = Path(plot_dir)
    csv_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)
    for alpha in sorted(df["Alpha"].unique()):
        tag = safe_alpha_name(alpha)
        sub = df[df["Alpha"] == alpha].sort_values(["side", "x"])
        sub.to_csv(csv_dir / f"curve_alpha_{tag}.csv", index=False)
        plot_cp_curve(sub, alpha, plot_dir / f"curve_alpha_{tag}.jpg", title_prefix=title_prefix)
